In [4]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay

# -----------------------------------------------------------------------------
# 1) Configuration
# -----------------------------------------------------------------------------
DATA_ROOT    = "data/resampled-v2"
RESULTS_ROOT = "results/Log_Reg"
SPLITS       = [
    ("2024-06-01 (80/20)", "ratio"),
    ("2024-10-01",        "2024-10-01"),
    ("2025-01-01",        "2025-01-01"),
]
INTERVALS    = ["1min", "10min", "1h", "1d"]
PARAM_GRID   = {"clf__C": [0.1, 1.0, 10.0]}
N_SPLITS_CV   = 5

# -----------------------------------------------------------------------------
# 2) Load & preprocess
# -----------------------------------------------------------------------------
def load_and_preprocess(path: str) -> pd.DataFrame:
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df.sort_values(["coin_id", "timestamp"], inplace=True)
    df["target_direction"] = (
        df.groupby("coin_id", observed=True)["close"]
          .shift(-1)
          .gt(df["close"])
          .astype(int)
    )
    df.dropna(subset=["target_direction"], inplace=True)
    return df

# -----------------------------------------------------------------------------
# 3) Feature engineering
# -----------------------------------------------------------------------------
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.sort_values(["coin_id", "timestamp"], inplace=True)
    df["volume_ratio"] = df["taker_buy_quote_asset_volume"] / df["quote_asset_volume"]
    df["log_ret"]      = df.groupby("coin_id")["close"].transform(lambda x: np.log(x).diff())
    df["range_pct"]    = (df["high"] - df["low"]) / df["open"]
    df["oc_pct"]       = (df["close"] - df["open"]) / df["open"]
    df["sma_10"]       = df.groupby("coin_id")["close"].transform(lambda x: x.rolling(10).mean())
    df["ema_10"]       = df.groupby("coin_id")["close"].transform(lambda x: x.ewm(span=10, adjust=False).mean())
    for w in [5, 10, 20, 50]:
        df[f"ret_ma_{w}"] = df.groupby("coin_id")["log_ret"].transform(lambda x: x.rolling(w).mean())
        df[f"vol_{w}"]    = df.groupby("coin_id")["log_ret"].transform(lambda x: x.rolling(w).std())
    df["mom_diff_5_50"] = df["ret_ma_5"] - df["ret_ma_50"]
    df["sma_7"]        = df.groupby("coin_id")["close"].transform(lambda x: x.rolling(7).mean())
    df["sma_30"]       = df.groupby("coin_id")["close"].transform(lambda x: x.rolling(30).mean())
    df["trend_ratio"]  = df["sma_7"] / df["sma_30"]
    sec = df["timestamp"].values.astype("int64") // 10**9
    day, week = 24*60*60, 7*24*60*60
    df["sin_day"]      = np.sin(2 * np.pi * sec / day)
    df["cos_day"]      = np.cos(2 * np.pi * sec / day)
    df["sin_week"]     = np.sin(2 * np.pi * sec / week)
    df["cos_week"]     = np.cos(2 * np.pi * sec / week)
    features = [
        "volume_ratio","log_ret","range_pct","oc_pct",
        "sma_10","ema_10","ret_ma_5","vol_5",
        "sma_7","sma_30","trend_ratio",
        "sin_day","cos_day","sin_week","cos_week"
    ]
    return df.dropna(subset=features)

# -----------------------------------------------------------------------------
# 4) Main Evaluation & Data Prep
# -----------------------------------------------------------------------------
os.makedirs(RESULTS_ROOT, exist_ok=True)
for split_label, dirname in SPLITS:
    split_dir = os.path.join(DATA_ROOT, dirname)
    for iv in INTERVALS:
        print(f"Processing {split_label} | {iv}")
        out_dir = os.path.join(RESULTS_ROOT, dirname, iv)
        os.makedirs(out_dir, exist_ok=True)
        train_df = add_features(load_and_preprocess(os.path.join(split_dir, f"train_{iv}.parquet")))
        test_df  = add_features(load_and_preprocess(os.path.join(split_dir, f"test_{iv}.parquet")))
        feature_cols = [
            c for c in train_df.columns if c not in ("coin_id","timestamp","target_direction")
        ]
        tscv = TimeSeriesSplit(n_splits=N_SPLITS_CV)
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("clf",     LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=1000))
        ])
        grid = GridSearchCV(pipe, param_grid=PARAM_GRID, cv=tscv, scoring="roc_auc", n_jobs=-1)
        for coin in train_df["coin_id"].unique():
            tr = train_df[train_df["coin_id"]==coin]
            te = test_df[test_df["coin_id"]==coin]
            if tr.empty or te.empty:
                continue
            X_tr, y_tr = tr[feature_cols], tr["target_direction"]
            X_te, y_te = te[feature_cols], te["target_direction"]
            grid.fit(X_tr, y_tr)
            best = grid.best_estimator_
            # generate signals
            df_te = te.sort_values("timestamp").reset_index(drop=True)
            preds = best.predict(df_te[feature_cols])
            signal = pd.Series(preds, index=df_te["timestamp"]).shift(1).fillna(0).astype(int)
            bar_ret = df_te.set_index("timestamp")["close"].pct_change().shift(-1).loc[signal.index]
            # assemble and save
            df_out = pd.DataFrame({"signal": signal, "bar_ret": bar_ret}).dropna()
            df_out.to_parquet(os.path.join(out_dir, f"{coin}.parquet"))

Processing 2024-06-01 (80/20) | 1min
Processing 2024-06-01 (80/20) | 10min
Processing 2024-06-01 (80/20) | 1h
Processing 2024-06-01 (80/20) | 1d
Processing 2024-10-01 | 1min
Processing 2024-10-01 | 10min


/opt/anaconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/opt/anaconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/opt/anaconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Processing 2024-10-01 | 1h


/opt/anaconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Processing 2024-10-01 | 1d
Processing 2025-01-01 | 1min
Processing 2025-01-01 | 10min


/opt/anaconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Processing 2025-01-01 | 1h
Processing 2025-01-01 | 1d
